# Module 5: Agent Frameworks
# Topics 35 & 36: Tools (`@tool`) & Tool Calling

> **Interview Difficulty:** ⭐⭐⭐⭐⭐ (Must Know)
>
> **Interview Frequency:** Extremely High
>
> **Prerequisites:**
> - Models ✅
> - Prompt Templates ✅
> - Messages ✅
> - LCEL ✅
> - Chains & Runnables ✅

---

# Learning Objectives

After this topic, you should be able to answer:

- What is a Tool?
- Why do LLMs need Tools?
- What is the `@tool` decorator?
- How to create custom tools
- How Tool Calling works
- bind_tools()
- AIMessage.tool_calls
- Tool execution flow
- Best Practices
- Interview Questions

---

# 1. Why Do We Need Tools?

An LLM only knows information from its training (plus any provided context). It **cannot**:

- Access live weather
- Query databases
- Call REST APIs
- Read local files
- Perform calculations reliably
- Execute Python code

To interact with the outside world, it needs **Tools**.

---

# Interview Definition ⭐⭐⭐⭐⭐

> **A Tool is a callable function that an LLM can invoke to perform external actions such as API calls, database queries, calculations, or file operations.**

---

# 2. LLM Without Tools

```text
User

↓

LLM

↓

Answer

(No external access)
```

Example

User:

```
What's the current weather in Pune?
```

The LLM can only guess or state it doesn't know.

---

# 3. LLM With Tools

```text
User

↓

LLM

↓

Weather Tool

↓

Weather API

↓

LLM

↓

Final Answer
```

The LLM decides that external information is required and requests the tool.

---

# 4. What is the `@tool` Decorator?

LangChain provides the `@tool` decorator to convert a Python function into a Tool.

Example

```python
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b
```

Now `multiply` is no longer just a Python function—it is a LangChain Tool with metadata.

---

# 5. Why Is the Docstring Important?

```python
@tool
def multiply(a: int, b: int):
    """Multiply two numbers."""
```

The docstring becomes the tool description shown to the model.

The LLM uses it to decide **when** to call the tool.

Poor description:

```
Does work
```

Good description:

```
Multiply two integer values.
```

---

# 6. Tool Schema

When a Tool is created, LangChain automatically generates a schema.

Example

```python
@tool
def add(a: int, b: int):
    """Add two integers."""
```

Internally

```json
{
  "name": "add",
  "description": "Add two integers.",
  "parameters": {
    "a": "integer",
    "b": "integer"
  }
}
```

This schema is sent to the model.

---

# 7. Creating Multiple Tools

```python
@tool
def add(a: int, b: int):
    """Add two numbers."""
    return a + b

@tool
def subtract(a: int, b: int):
    """Subtract two numbers."""
    return a - b

tools = [add, subtract]
```

---

# 8. What is Tool Calling?

Tool Calling is the ability of an LLM to:

1. Decide whether a Tool is needed.
2. Select the correct Tool.
3. Generate arguments.
4. Request tool execution.

The model **does not execute** the tool itself.

It requests the application to execute it.

---

# Interview Definition ⭐⭐⭐⭐⭐

> **Tool Calling is the process where an LLM determines that an external function is required, selects the appropriate tool, generates its arguments, and requests the application to execute it.**

---

# 9. Tool Calling Lifecycle

```text
User

↓

LLM

↓

Tool Decision

↓

Generate Arguments

↓

Application Executes Tool

↓

Tool Result

↓

LLM

↓

Final Answer
```

---

# 10. Registering Tools

```python
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini"
)

llm_with_tools = llm.bind_tools(tools)
```

`bind_tools()` informs the model which tools are available.

---

# 11. Example Conversation

User

```
What is 25 × 8?
```

Model thinks:

```
I should call multiply().
```

Generated request

```json
{
    "tool": "multiply",
    "args": {
        "a": 25,
        "b": 8
    }
}
```

The application executes:

```python
multiply(25, 8)
```

Returns

```
200
```

The result is sent back to the model.

Final response

```
25 × 8 = 200
```

---

# 12. What Does the Model Return?

The first response is **not** always the final answer.

It may contain tool requests.

Example

```python
response = llm_with_tools.invoke(
    "Multiply 10 and 15."
)
```

Check:

```python
response.tool_calls
```

Example output

```python
[
    {
        "name": "multiply",
        "args": {
            "a": 10,
            "b": 15
        }
    }
]
```

---

# 13. Complete Tool Execution Flow

```text
Human

↓

Chat Model

↓

AIMessage

↓

tool_calls

↓

Python Function

↓

Result

↓

ToolMessage

↓

LLM

↓

Final AIMessage
```

---

# 14. Complete Coding Example

```python
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b

llm = ChatOpenAI(
    model="gpt-4o-mini"
)

llm = llm.bind_tools([multiply])

response = llm.invoke(
    "Multiply 12 and 8."
)

print(response.tool_calls)
```

Notice:

The model requests the tool. Your application still needs to execute it and return the result.

---

# 15. Important Interview Concept

### Does the LLM execute the tool?

**No.**

The LLM only produces a structured tool call.

Your application (or an agent framework like LangGraph) executes the tool and sends the result back to the model.

---

# 16. Tool Calling Architecture

```text
User

↓

Chat Model

↓

Tool Schema

↓

Tool Selection

↓

Arguments Generated

↓

Application

↓

Tool Execution

↓

Tool Result

↓

LLM

↓

Final Answer
```

---

# 17. When Does the LLM Choose a Tool?

It decides based on:

- User query
- Tool description
- Tool parameter schema
- System instructions
- Conversation context

Example

```
"What is 5 + 8?"
```

Calls Calculator Tool.

---

```
"Read employee.pdf"
```

Calls File Tool.

---

```
"What's today's weather?"
```

Calls Weather Tool.

---

# 18. Common Enterprise Tools

- Weather API
- SQL Database
- Search Engine
- Python REPL
- Vector Database
- Email Sender
- Calendar
- File Reader
- CRM
- Jira
- GitHub

---

# 19. Best Practices

✅ Write clear tool docstrings.

✅ Keep tools focused on one responsibility.

✅ Validate tool inputs.

✅ Handle tool failures gracefully.

✅ Return structured outputs when possible.

---

# 20. Common Mistakes

❌ Assuming the model executes tools.

❌ Writing vague tool descriptions.

❌ Creating one tool that performs many unrelated tasks.

❌ Forgetting to bind tools before invoking the model.

---

# 21. Tools vs Tool Calling

| Tools | Tool Calling |
|--------|--------------|
| External functions | Decision to invoke a tool |
| Implemented by developer | Performed by the LLM |
| Execute business logic | Generates tool requests |
| Return results | Produces arguments |

---

# 22. Tool Calling vs Function Calling

This is a popular interview question.

| Function Calling | Tool Calling |
|------------------|--------------|
| Provider feature (OpenAI, Anthropic, etc.) | LangChain abstraction |
| Model emits function call | Model emits tool call |
| Provider-specific APIs | Unified LangChain interface |

In practice, LangChain's Tool Calling uses the provider's underlying function-calling capability.

---

# 23. Interview Questions

## Q1. What is a Tool?

**Answer:**

A Tool is a callable function exposed to an LLM so it can perform external actions such as API calls, calculations, database queries, or file operations.

---

## Q2. What does `@tool` do?

**Answer:**

It converts a Python function into a LangChain Tool by generating metadata such as the tool name, description, and parameter schema.

---

## Q3. Does the LLM execute tools?

**Answer:**

No. The LLM only decides which tool to use and generates the required arguments. The application executes the tool and returns the result to the model.

---

## Q4. What does `bind_tools()` do?

**Answer:**

It registers available tools with the chat model so the model knows which tools it is allowed to call.

---

## Q5. How does the model decide which tool to call?

**Answer:**

It uses the tool descriptions, parameter schema, user query, conversation context, and system instructions to determine the most appropriate tool.

---

## Q6. What is the difference between Tools and Tool Calling?

**Answer:**

Tools are the executable functions created by developers, while Tool Calling is the model's process of selecting a tool and generating arguments for it.

---

# 24. Quick Revision

| Concept | Purpose |
|----------|---------|
| Tool | External function |
| @tool | Converts Python function into a Tool |
| bind_tools() | Registers tools with the model |
| tool_calls | Tool requests generated by the model |
| ToolMessage | Sends tool results back to the model |

---

# Interview Cheat Sheet

```text
User

↓

LLM

↓

Tool Selection

↓

tool_calls

↓

Application

↓

Execute Tool

↓

Tool Result

↓

ToolMessage

↓

LLM

↓

Final Response

Key APIs

@tool

bind_tools()

response.tool_calls
```

---

# 30-Second Interview Answer

> **In LangChain, a Tool is a Python function exposed to an LLM using the `@tool` decorator. During Tool Calling, the model decides whether an external action is needed, selects the appropriate tool, and generates its arguments. The application executes the tool and returns the result to the model, which then produces the final response. This separation keeps the LLM focused on reasoning while external systems handle real-world actions.**

---

# Key Takeaway

> **Tools give an LLM capabilities beyond text generation. The model reasons about *what* needs to be done, while the application performs *how* it is done by executing the selected tool and feeding the result back into the conversation.**